<a href="https://colab.research.google.com/github/IBIZAHUB/BIT4133-Week7-Neural-Language-Models/blob/main/BIT4133_Week7_Neural_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical Task 3: TensorFlow NLP Exercise

This task focuses on fundamental Natural Language Processing (NLP) steps using TensorFlow's Keras API, including text input, tokenization, displaying word indices, and converting text to sequences.

## 1. Accept text input

First, we'll define a sample text string that will be used for all subsequent NLP operations in this task. You can modify this string to experiment with different inputs.

In [ ]:
# Define the text input
input_text = "TensorFlow is an open-source machine learning platform. It has a comprehensive, flexible ecosystem of tools, libraries, and community resources."

print(f"Input Text: {input_text}")

Input Text: TensorFlow is an open-source machine learning platform. It has a comprehensive, flexible ecosystem of tools, libraries, and community resources.


## 2. Tokenize the text and display word indices

We'll use TensorFlow's `Tokenizer` to convert the text into a sequence of numbers. This involves building a vocabulary from the text and then mapping each word to a unique integer. We will then display the generated word indices.

In [ ]:
import tensorflow as tf

# Initialize the Tokenizer
task3_tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=None, oov_token='<unk>')

# Fit the tokenizer on the input text
task3_tokenizer.fit_on_texts([input_text])

# Display word indices
word_index_task3 = task3_tokenizer.word_index
print("\nWord Index:")
for word, index in list(word_index_task3.items())[:15]: # Displaying first 15 for brevity
    print(f"'{word}': {index}")

print(f"\nTotal unique words (including OOV token if present): {len(word_index_task3) + 1}")


Word Index:
'<unk>': 1
'tensorflow': 2
'is': 3
'an': 4
'open': 5
'source': 6
'machine': 7
'learning': 8
'platform': 9
'it': 10
'has': 11
'a': 12
'comprehensive': 13
'flexible': 14
'ecosystem': 15

Total unique words (including OOV token if present): 22


## 3. Convert text into sequences

Finally, we will convert the raw input text into sequences of integers. Each word in the text will be replaced by its corresponding integer ID from the `word_index` created by the tokenizer. This is the numerical representation that machine learning models typically process.

In [ ]:
# Convert text to sequences of integers
text_sequences_task3 = task3_tokenizer.texts_to_sequences([input_text])

print("\nText Sequences:")
print(text_sequences_task3)

print(f"\nOriginal Text: {input_text}")

# To verify, let's convert the first sequence back to words
reverse_word_index_task3 = dict([(index, word) for word, index in word_index_task3.items()])

def sequence_to_text(list_of_indices):
    return ' '.join([reverse_word_index_task3.get(index, '<unk>') for index in list_of_indices])

print(f"\nConverted sequence back to text: {sequence_to_text(text_sequences_task3[0])}")


Text Sequences:
[[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]]

Original Text: TensorFlow is an open-source machine learning platform. It has a comprehensive, flexible ecosystem of tools, libraries, and community resources.

Converted sequence back to text: tensorflow is an open source machine learning platform it has a comprehensive flexible ecosystem of tools libraries and community resources


## 1. Create a small dataset

We'll start by defining a simple text corpus that will serve as our dataset for training the text prediction model. This dataset should be large enough to demonstrate the concept but small enough to be manageable for this practical task.

In [ ]:
text = """\nThere are many variations of passages of Lorem Ipsum available, but the majority have suffered alteration in some form, by injected humour, or randomised words which don't look even slightly believable. If you are going to use a passage of Lorem Ipsum, you need to be sure there isn't anything embarrassing hidden in the middle of text. All the Lorem Ipsum generators on the Internet tend to repeat predefined chunks as necessary, making this the first true generator on the Internet. It uses a dictionary of over 200 Latin words, combined with a handful of model sentence structures, to generate Lorem Ipsum which looks reasonable. The generated Lorem Ipsum is therefore always free from repetition, injected humour, or non-characteristic words etc.\n"""

print(f"Dataset created with {len(text)} characters.")

Dataset created with 752 characters.


## 2. Tokenize text

To prepare our text data for a neural network, we need to tokenize it. This process involves two main steps:
1.  **Splitting text into words**: Breaking down the corpus into individual words.
2.  **Mapping words to integers**: Assigning a unique integer to each word, creating a numerical representation of the text.

We will use `tf.keras.preprocessing.text.Tokenizer` for this purpose, which can handle both tokenization and converting text to sequences of integers.

In [ ]:
import tensorflow as tf

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
total_words = len(word_index) + 1

print(f"Total unique words: {total_words}")
print(f"First 10 words in vocabulary: {list(word_index.keys())[:10]}")

# Convert text to sequence of integers
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total input sequences generated: {len(input_sequences)}")
print(f"Example input sequence: {input_sequences[0]}")

Total unique words: 87
First 10 words in vocabulary: ['the', 'of', 'lorem', 'ipsum', 'to', 'words', 'a', 'there', 'are', 'in']
Total input sequences generated: 121
Example input sequence: [8, 9]


## 3. Train a neural network

Now that our text is tokenized and converted into sequences, we can prepare the data for training a neural network. This involves:
1.  **Padding sequences**: Ensuring all input sequences have the same length.
2.  **Creating features (X) and labels (y)**: The input sequences will be our features, and the last word of each sequence will be the label (the word we want to predict).
3.  **One-hot encoding labels**: Converting the integer labels into a one-hot encoded format.
4.  **Defining the model**: Using a `tf.keras.Sequential` model with an `Embedding` layer, `LSTM` layer, and a `Dense` output layer.
5.  **Compiling and training the model**.

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Find the maximum sequence length for padding
max_sequence_len = max([len(x) for x in input_sequences])

# Pad sequences
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    input_sequences, maxlen=max_sequence_len, padding='pre'
)

# Split into features (X) and labels (y)
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

# One-hot encode the labels
y = to_categorical(y, num_classes=total_words)

print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (labels): {y.shape}")

# Define the model
model = Sequential()
model.add(Embedding(total_words, 10)) # Removed input_length
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))

# Explicitly build the model to show parameter counts in summary
model.build(input_shape=(None, max_sequence_len - 1))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Model Summary:")
model.summary()

# Train the model (for demonstration, a small number of epochs)
print("\nTraining the model...")
model.fit(X, y, epochs=50, verbose=1)

Shape of X (features): (121, 121)
Shape of y (labels): (121, 87)
Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 121, 10)        │           870 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 150)            │        96,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 87)             │        13,137 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,607 (432.06 KB)

 Trainable params: 110,607 (432.06 KB)

 Non-trainable params: 0 (0.00 B)


Training the model...
Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.0413 - loss: 4.4664
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.0331 - loss: 4.4590
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step - accuracy: 0.0496 - loss: 4.4456
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step - accuracy: 0.0496 - loss: 4.3278
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 156ms/step - accuracy: 0.0496 - loss: 4.3730
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 227ms/step - accuracy: 0.0413 - loss: 4.2994
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 307ms/step - accuracy: 0.0579 - loss: 4.2855
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 248ms/step - accuracy: 0.0579 - loss: 4.2831
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 149ms/step - accuracy: 0.0579 - loss: 4.2770
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.0579 - loss: 4.2684
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step - accuracy: 0.0579 - loss: 4.2660
Epoch 12/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step 

## 4. Predict the next word

With our trained neural network, we can now predict the next word in a sequence. This involves:
1.  **Preprocessing the input sentence**: Tokenizing and padding the input text similar to how the training data was prepared.
2.  **Using the model to predict probabilities**: The model will output probabilities for each word in the vocabulary.
3.  **Selecting the word with the highest probability**: Converting the predicted integer back to a word using the tokenizer's `word_index`.

In [ ]:
def predict_next_word(seed_text, n_words):
    for _ in range(n_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=max_sequence_len - 1, padding='pre'
        )
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probs)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# Example usage:
seed = "lorem ipsum"
predicted_sentence = predict_next_word(seed, 5)
print(f"Seed text: '{seed}'")
print(f"Predicted next 5 words: '{predicted_sentence}'")

seed = "there are"
predicted_sentence = predict_next_word(seed, 3)
print(f"\nSeed text: '{seed}'")
print(f"Predicted next 3 words: '{predicted_sentence}'")

Seed text: 'lorem ipsum'
Predicted next 5 words: 'lorem ipsum of of of of ipsum'

Seed text: 'there are'
Predicted next 3 words: 'there are of of of'


In [ ]:
def predict_next_words_with_confidence(seed_text, num_predictions=1):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = tf.keras.preprocessing.sequence.pad_sequences(
        [token_list], maxlen=max_sequence_len - 1, padding='pre'
    )

    predicted_probs = model.predict(token_list, verbose=0)[0]

    # Get top N predictions
    top_n_indices = np.argsort(predicted_probs)[::-1][:num_predictions]

    results = []
    for idx in top_n_indices:
        confidence = predicted_probs[idx]
        word = ""
        for w, i in tokenizer.word_index.items():
            if i == idx:
                word = w
                break
        if word:
            results.append((word, confidence))
    return results


### Interactive Predictive Text Application

Now, let's create a simple interactive interface where you can input a sentence fragment and see the predicted next words along with their confidence scores.

In [ ]:
print("\n--- Mini Predictive Text Application ---")
print("Enter a sentence fragment to predict the next word(s).")
print("Type 'quit' to exit.\n")

while True:
    user_input = input("Your fragment: ")
    if user_input.lower() == 'quit':
        break
    if not user_input.strip():
        print("Please enter some text.")
        continue

    num_preds_to_show = 3 # Display top 3 predictions
    predictions = predict_next_words_with_confidence(user_input, num_predictions=num_preds_to_show)

    if predictions:
        print(f"Predicted next word(s) for '{user_input}':")
        for word, confidence in predictions:
            print(f"  - '{word}' (Confidence: {confidence:.4f})")
    else:
        print("No predictions could be made (perhaps input words are not in vocabulary).")

print("Application closed.")



--- Mini Predictive Text Application ---
Enter a sentence fragment to predict the next word(s).
Type 'quit' to exit.

Predicted next word(s) for 'i ':
  - 'are' (Confidence: 0.2387)
  - 'of' (Confidence: 0.2281)
  - 'many' (Confidence: 0.2251)
